In [1]:
import numpy as np
import spikecorec as spc

In [2]:
print(spc.__version__)

0.1.0


In [3]:

def format_bytes(byte_count: int) -> str:
    value = float(byte_count)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if value < 1024 or unit == "TiB":
            return f"{value:.1f} {unit}"
        value /= 1024

In [ ]:
N = 50
lifetime = 100000
record_stride = 11
network = spc.square_torus(N)
engine = spc.SpikeEngine(
    network, 
    shape=[N, N], 
    rank=64, 
    decay_rate = 0.949
)


In [5]:
w_accum, w_instant = engine.estimate_bifurcation_weight(input_period=1)
target, _, _ = engine.scale_uniform_weights_near_bifurcation(input_period=1, scale = 1.052, freeze_learning=True)
print(f"w_accum={w_accum:.6f} w_instant={w_instant:.6f} target={target:.6f}")
print("constant weights are enabled for this static propagation benchmark only")

w_accum=0.854100 w_instant=0.900000 target=0.898513
constant weights are enabled for this static propagation benchmark only


In [6]:
input_neuron = (N * N) // 2 + N // 2
engine.set_input_neurons([input_neuron])
input_spikes = np.ones((lifetime, 1), dtype = np.float32).tolist()
recorded_frames = (lifetime + record_stride - 1) // record_stride
estimated_bytes = N * N * recorded_frames * np.dtype(np.float32).itemsize
print(f"streaming {recorded_frames} frames, uncompressed membrane payload {format_bytes(estimated_bytes)}")

engine.start_static_record(
    input_spikes,
    lifetime,
    "metal_test_10.spire.gz",
    record_membrane = True,
    full_decay = True,
    compression_level = 4,
    compression_async = True,
    record_stride = record_stride
)

streaming 4546 frames, uncompressed membrane payload 43.4 MiB


In [7]:
index = 1000
print(f"neighbors: {engine.weights.get_neighbors(index)}")

recorded_frames = (lifetime + record_stride - 1) // record_stride
estimated_bytes = N * N * recorded_frames * np.dtype(np.float32).itemsize
print(format_bytes(estimated_bytes))

neighbors: [950, 1001, 1049, 1050]
43.4 MiB
